In [ ]:
import cv2
import numpy as np
import os
import glob
import time

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((10*7,3), np.float32)
objp[:,:2] = np.mgrid[0:10,0:7].T.reshape(-1,2)
# compensate for square size
square_size = 0.022
objp *= square_size
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

In [ ]:
cap = cv2.VideoCapture(0)

os.makedirs('images', exist_ok=True)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (10,7), None)

    if ret:
        objpoints.append(objp)

        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        cv2.imwrite("images/calib_%f.jpg" % time.time(), frame)
        cv2.drawChessboardCorners(frame, (10,7), corners2, ret)

    cv2.imshow("Calibration", frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
images = glob.glob('images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (10,7), None)
 
    # If found, add object points, image points (after refining them)
    if ret:
        objpoints.append(objp)
 
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

In [ ]:
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

In [ ]:
os.makedirs('calibration', exist_ok=True)
np.savetxt('calibration/calib_mtx.csv', mtx, delimiter=',')
np.savetxt('calibration/calib_dist.csv', dist, delimiter=',')